# 📊 Task 2: End-to-End ML Pipeline — Telco Customer Churn

**Objective:** Build a reusable, production-ready ML pipeline to predict customer churn using scikit-learn's `Pipeline` API.

| Step | Description |
|------|-------------|
| 1 | Load & explore the Telco Churn dataset |
| 2 | Clean & prepare data |
| 3 | Define preprocessing (scaling + encoding) |
| 4 | Build sklearn Pipeline |
| 5 | Tune with GridSearchCV (Logistic Regression + Random Forest) |
| 6 | Evaluate on test set |
| 7 | Export pipeline with joblib |
| 8 | Feature importance (if Random Forest wins) |

## 📦 Install Dependencies

In [ ]:
!pip install pandas numpy scikit-learn joblib -q

## 📥 Step 1 — Load and Explore Dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score, precision_score, recall_score
)
import joblib
import warnings
warnings.filterwarnings('ignore')

url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

print('Dataset shape:', df.shape)
df.head()

In [ ]:
print('Data types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())

## 🧹 Step 2 — Clean and Prepare Data

In [ ]:
# TotalCharges has blank strings for brand-new customers -> convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])

# Drop customerID - not a predictive feature
df = df.drop('customerID', axis=1)

# Encode target: Yes -> 1, No -> 0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print('Cleaned dataset shape:', df.shape)
print('\nTarget distribution:')
print(df['Churn'].value_counts(normalize=True).rename({0: 'No Churn', 1: 'Churn'}))

## ✂️ Step 3 — Train/Test Split

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set size: {X_train.shape[0]}')
print(f'Test set size:     {X_test.shape[0]}')

## ⚙️ Step 4 — Define Preprocessing Pipeline

In [ ]:
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print('Numeric columns    :', numeric_cols)
print('Categorical columns:', categorical_cols)

# drop='first' avoids dummy variable trap in Logistic Regression
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_cols)
    ],
    remainder='drop'
)

print('\nPreprocessor built successfully.')

## 🤖 Step 5 — Build Pipeline and Tune with GridSearchCV

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Both models in a single GridSearchCV — correct production approach
param_grid = [
    {
        'classifier': [LogisticRegression(random_state=42, max_iter=1000)],
        'classifier__C': [0.1, 1.0, 10.0],
        'classifier__solver': ['liblinear', 'saga']
    },
    {
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [50, 100, 200],
        'classifier__max_depth': [None, 10, 20],
        'classifier__min_samples_split': [2, 5]
    }
]

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

print('Starting GridSearchCV...')
grid_search.fit(X_train, y_train)

print('\nBest parameters found:')
print(grid_search.best_params_)
print(f'Best CV F1-score: {grid_search.best_score_:.4f}')

## 📊 Step 6 — Evaluate on Test Set

In [ ]:
best_pipeline = grid_search.best_estimator_
y_pred  = best_pipeline.predict(X_test)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]

print('=' * 50)
print('TEST SET PERFORMANCE')
print('=' * 50)
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'F1-Score : {f1_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_proba):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

## 💾 Step 7 — Export Pipeline with joblib

In [ ]:
model_filename = 'churn_pipeline.joblib'
joblib.dump(best_pipeline, model_filename)
print(f'Pipeline exported to: {model_filename}')
print('Contains: preprocessing steps + trained model — single reusable artifact.')

# Reload and predict — proves it works standalone
loaded_pipeline = joblib.load(model_filename)
sample_pred  = loaded_pipeline.predict(X_test.iloc[[0]])[0]
sample_proba = loaded_pipeline.predict_proba(X_test.iloc[[0]])[0][1]
print(f'\nReload test -> Prediction: {"Churn" if sample_pred == 1 else "No Churn"} '
      f'(churn probability: {sample_proba:.2%})')
print(f'Actual label:              {"Churn" if y_test.iloc[0] == 1 else "No Churn"}')

## 🌳 Step 8 — Feature Importance (if Random Forest won)

In [ ]:
if isinstance(best_pipeline.named_steps['classifier'], RandomForestClassifier):
    preprocessor_fitted = best_pipeline.named_steps['preprocessor']
    cat_encoder  = preprocessor_fitted.named_transformers_['cat']
    cat_columns  = cat_encoder.get_feature_names_out(categorical_cols)
    all_features = numeric_cols + list(cat_columns)

    importances = best_pipeline.named_steps['classifier'].feature_importances_
    feature_df  = pd.DataFrame({
        'feature': all_features,
        'importance': importances
    }).sort_values('importance', ascending=False)

    print('Top 10 most important features:')
    print(feature_df.head(10).to_string(index=False))
else:
    print('Best model is Logistic Regression — feature importance not applicable.')
    print('(For LR, you can inspect coefficients via best_pipeline.named_steps["classifier"].coef_)')

print('\nTask 2 completed successfully.')